# Intraday JKM-TTF vs Arb Breakevens

Here we plot JKM-TTF prices against relevant Arb Breakeven levels. This script can also be used for TTF prices vs TTF breakevens.

For a full explanation of how to import our Intraday Contracts or Breakevens data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html

### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint:
https://www.sparkcommodities.com/api/

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

This section can remain unchanged for most Spark API users.

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.

The code then prints the available prices that are callable from the API, and their corresponding Python ticker names are displayed as a list at the bottom of the Output.

In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

# 2. Arb Breakevens Reference Data

The Reference Data endpoint returns the UUID codes associated with each FoB Port available on the Arb Breakevens endpoint. 

If there are two arb breakevens available for a single FoB Port, e.g. "Sabine Pass via COGH" or "Sabine Pass via Panama", these are included as separate entries in the Reference Data response. The FoB Port UUID remains the same between these two entries. 

In [ ]:
# define reference data function

def fetch_reference_data(access_token, format='json'):

    uri="/v1.0/intraday/breakevens/reference-data/"
    print(uri)
    
    content = do_api_get_query(
        uri,  access_token=access_token, format=format
    )

    if format == 'json':
        data = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            data = pd.DataFrame()
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))  # automatically converting into a Pandas DataFrame when choosing CSV format

    return data


# call data
ref_df = fetch_reference_data(access_token, format='csv')
ref_df

# 3. Endpoint Fetch Functions

## 3.1 Arb Breakevens

Details on endpoint parameters can be found on the API docs website:

https://www.sparkcommodities.com/api/intraday/breakevens.html

In [ ]:
## Defining the function


def fetch_historical_breakevens(access_token, breakeven_type, ticker, via, percent_hire, unit, extended_curve, start, end, format='json'):
    
    query_params = "?breakeven-type={}".format(breakeven_type)
    query_params += "&fob-port={}".format(ticker)
    query_params += "&via-point={}".format(via)
    query_params += "&percent-hire={}".format(percent_hire)
    query_params += "&unit={}".format(unit)
    query_params += "&extended-curve={}".format(extended_curve)
    

    query_params += "&start={}".format(start)
    query_params += "&end={}".format(end)

    uri="/v1.0/intraday/breakevens/historical/{}".format(query_params)
    print(uri)
    
    content = do_api_get_query(
        uri=uri, access_token=access_token, format=format
    )

    if format == 'json':
        data = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            data = pd.DataFrame()
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))  # automatically converting into a Pandas DataFrame when choosing CSV format
            data['AsOf'] = pd.to_datetime(data['AsOf'], utc=True).dt.tz_localize(None)

    return data


## 3.2 Contracts Data (JKM-TTF or TTF)

This endpoint takes 4 required parameters - details on these parameters can be found on our API docs website:

https://www.sparkcommodities.com/api/intraday/contracts.html

In [ ]:
## Defining the function

def fetch_historical_contracts(access_token, contract, unit, start, end, format='json'):
    
    query_params = "?contract={}".format(contract)
    query_params += "&unit={}".format(unit)

    query_params += "&start={}".format(start)
    query_params += "&end={}".format(end)

    uri="/v1.0/intraday/contracts/historical/{}".format(query_params)
    print(uri)
    
    content = do_api_get_query(
        uri=uri, access_token=access_token, format=format
    )

    if format == 'json':
        data = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            data = pd.DataFrame()
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))  # automatically converting into a Pandas DataFrame when choosing CSV format
            data['AsOf'] = pd.to_datetime(data['AsOf'], utc=True).dt.tz_localize(None)

    return data


# 4. Historical Data Function

Currently, a maximum of 20 historical days can be called at one time due to the size of the data file. If more data points are required, the below code can be used, which splits the requested date range into looped calls of 20 historical days at a time.

This function is applicable to both the Contracts and Breakevens endpoints - when calling the function, just specify which endpoint ('contracts' or 'breakevens') it's being used for. 

The 'start', 'end' and 'unit' parameters are applicable to both endpoints. All other parameters are contract specific - please refer to the documentation or the api_fetch functions above to see which parameters are applicable to each endpoint

In [ ]:
def loop_historical_data(access_token, endpoint, contract=None, breakeven_type=None, ticker=None, via=None, 
                        percent_hire=None, unit=None, extended_curve=None, hist_start=None, hist_end=None):
    
    # error handling
    if endpoint not in {"breakevens", "contracts"}:
        raise ValueError("endpoint must be either 'breakevens' or 'contracts'")
    if not hist_start or not hist_end:
        raise ValueError("hist_start and hist_end are required (YYYY-MM-DD)")

    start_dt = datetime.datetime.strptime(hist_start, '%Y-%m-%d')
    end_dt = datetime.datetime.strptime(hist_end, '%Y-%m-%d')
    if end_dt < start_dt:
        raise ValueError("hist_end must be on or after hist_start")

    # calculating the amount of days to retrieve data for
    hist_diff = (end_dt - start_dt).days
    
    t = 0

    starts = []
    ends = []
    
    # iterating through the historical data, 20 days at a time
    while t < hist_diff:
        # first loop & initialising the historical Dataframe
        if t == 0 and hist_diff >= 21:
            diff_end = datetime.datetime.strftime(datetime.datetime.strptime(hist_start, '%Y-%m-%d') + pd.Timedelta(days=20), '%Y-%m-%d')
            if endpoint == 'breakevens':
                hist_df = fetch_historical_breakevens(access_token, breakeven_type=breakeven_type, ticker=ticker, via=via, 
                                                    percent_hire=percent_hire, unit=unit, extended_curve=extended_curve, 
                                                        start=hist_start, end=diff_end, format='csv')
            elif endpoint == 'contracts':
                hist_df = fetch_historical_contracts(access_token, contract=contract, unit=unit, start=hist_start, end=diff_end, format='csv')

            starts.append(hist_start)
            ends.append(diff_end)
        
        # in case this function is used for a date range of < 20 days
        elif t==0 and hist_diff < 21:
            if endpoint == 'breakevens':
                hist_df = fetch_historical_breakevens(access_token, breakeven_type=breakeven_type, ticker=ticker, via=via, 
                                                    percent_hire=percent_hire, unit=unit, extended_curve=extended_curve, 
                                                    start=hist_start, end=hist_end, format='csv')
            elif endpoint == 'contracts':
                hist_df = fetch_historical_contracts(access_token, contract=contract, unit=unit, start=hist_start, end=hist_end, format='csv')

        # all other loops
        else:
            # calling data for the next 20 days
            if t < hist_diff-20:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(hist_start, '%Y-%m-%d') + pd.Timedelta(days=t), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=20), '%Y-%m-%d')
                if endpoint == 'breakevens':
                    hist2 = fetch_historical_breakevens(access_token, breakeven_type=breakeven_type, ticker=ticker, via=via, 
                                                        percent_hire=percent_hire, unit=unit, extended_curve=extended_curve,
                                                        start=diff_start, end=diff_end, format='csv')
                elif endpoint == 'contracts':
                    hist2 = fetch_historical_contracts(access_token, contract=contract, unit=unit, start=diff_start, end=diff_end, format='csv')
                hist_df = pd.concat([hist_df,hist2])
                starts.append(hist_start)
                ends.append(diff_end)
            
            # if <20 days until the end of the date range
            else:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(hist_start, '%Y-%m-%d') + pd.Timedelta(days=t), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=hist_diff-t), '%Y-%m-%d')
                if endpoint == 'breakevens':
                    hist2 = fetch_historical_breakevens(access_token, breakeven_type=breakeven_type, ticker=ticker, via=via, 
                                                        percent_hire=percent_hire, unit=unit, extended_curve=extended_curve,
                                                        start=diff_start, end=diff_end, format='csv')
                elif endpoint == 'contracts':
                    hist2 = fetch_historical_contracts(access_token, contract=contract, unit=unit, start=diff_start, end=diff_end, format='csv')
                hist_df = pd.concat([hist_df,hist2])
                starts.append(hist_start)
                ends.append(diff_end)
        
        t += 20

    hist_df['Value'] = pd.to_numeric(hist_df['Value'])
    hist_df['AsOf'] = pd.to_datetime(hist_df['AsOf'], utc=True).dt.tz_localize(None)

    return hist_df

# 5. Calling the Historical Data

In [ ]:
# Save your chosen FoB Port UUIDs as a variable
sab_uuid = ref_df[ref_df['FobPortName'] == 'Sabine Pass']['FobPortUuid'].iloc[0]
bon_uuid = ref_df[ref_df['FobPortName'] == 'Bonny LNG']['FobPortUuid'].iloc[0]

In [ ]:
# Calling the extended historical data function

start = '2025-10-01'
today = datetime.date.today().strftime('%Y-%m-%d')
#end = today
end= '2026-06-01'

# Sabine Pass Arb Breakeven
sab_df = loop_historical_data(access_token, endpoint='breakevens', breakeven_type='jkm-ttf', ticker=sab_uuid, via='cogh', 
                                        percent_hire=100, unit='usd-per-mmbtu', extended_curve=True,
                                         hist_start=start, hist_end=end)

# Sabine Pass Arb (Freight Considered Sunk) Breakeven
sab0_df = loop_historical_data(access_token, endpoint='breakevens', breakeven_type='jkm-ttf', ticker=sab_uuid, via='cogh', 
                                        percent_hire=0, unit='usd-per-mmbtu', extended_curve=True,
                                         hist_start=start, hist_end=end)

# Bonny LNG Arb Breakeven
bon_df = loop_historical_data(access_token, endpoint='breakevens', breakeven_type='jkm-ttf', ticker=bon_uuid, via='cogh', 
                                        percent_hire=100, unit='usd-per-mmbtu', extended_curve=True,    
                                         hist_start=start, hist_end=end)

# JKM-TTF prices
con_df = loop_historical_data(access_token, endpoint='contracts', contract='jkm-ttf', unit='usd-per-mmbtu', hist_start=start, hist_end=today)

sab_df.head(5)


In [ ]:
# Combining the contract data & the breakevens data into a single dataframe

# listing all breakevens Dataframes in a list
break_dataframes = [sab_df, bon_df, sab0_df]
break_names = ['Sabine Pass', 'Bonny LNG', 'Sabine Pass Sunk Freight']

# initialising dataframe with the contract data
combined_df = con_df.copy()
combined_df = combined_df.rename(columns={'Value':'ContractValue'})

# merging with the breakevens dataframes
dc = 0
for df in break_dataframes:
    df = df.rename(columns={'Value':break_names[dc]})
    combined_df = pd.merge(combined_df, df[['AsOf', 'PeriodName', break_names[dc]]], on=['AsOf', 'PeriodName'], how='left')
    dc+=1

# defining "ReleaseDay" so that data can be filtered by day if needed
combined_df['ReleaseDay'] = combined_df['AsOf'].dt.strftime('%Y-%m-%d')

In [ ]:
combined_df

# 6. Plotting

In [ ]:
# define which month contract to look at
month = 'Jun26'

# create dataframe by filtering combined_df
month_df =  combined_df[combined_df['PeriodName'] == month].copy()

# creating columns to show the difference between the Contract price and the Breakeven levels
month_df['Sabine Pass - Breakeven Diff'] = month_df['ContractValue'] - month_df['Sabine Pass']
month_df['Bonny LNG - Breakeven Diff'] = month_df['ContractValue'] - month_df['Bonny LNG']

month_df

In [ ]:
# Filter to trading hours & weekdays
month_p = month_df.copy()
month_p = month_p[pd.to_datetime(month_p['AsOf']).dt.dayofweek < 5]           # drop weekends
month_p = month_p[(pd.to_datetime(month_p['AsOf']).dt.hour >= 7) &
                   (pd.to_datetime(month_p['AsOf']).dt.hour < 17)]             # 07:00–17:00 UKT
month_p = month_p.sort_values('AsOf').reset_index(drop=True)
x = month_p.index

# Build x-tick positions
date_tick_idx = (
    month_p.groupby(pd.to_datetime(month_p['AsOf']).dt.normalize())
    .apply(lambda g: g.index[0])
)
tick_positions = date_tick_idx.values
tick_labels    = [d.strftime('%d %b') for d in date_tick_idx.index]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

fig, ax = plt.subplots(figsize=(14, 7))
fig.subplots_adjust(top=0.78, bottom=0.10, left=0.06, right=0.96)

ax.axhline(0, color='grey', alpha=0.85, zorder=0)

ax.plot(x, month_p['ContractValue'],        linewidth=2.0, color='#dc853f', label=month + ' JKM-TTF')
ax.plot(x, month_p['Sabine Pass'],   linewidth=2.0, color='#008bcc', label='Sabine Pass (via COGH) Arb Breakeven')
ax.plot(x, month_p['Bonny LNG'], linewidth=2.0, color='firebrick', label='Bonny LNG Arb Breakeven')

# Shading green when the US arb is open (JKM-TTF > Sabine Pass COGH Breakeven)
ax.fill_between(x, month_p['ContractValue'], month_p['Sabine Pass'],
                where=(month_p['Sabine Pass - Breakeven Diff'] > 0),
                color='mediumseagreen', alpha=0.2)

# Shading red when the Panama arb is closed (JKM-TTF < Sabine Pass Panama Breakeven)
ax.fill_between(x, month_p['Bonny LNG'], month_p['ContractValue'],
                where=(month_p['Bonny LNG - Breakeven Diff'] < 0),
                color='red', alpha=0.1)

plt.xlim([month_df['AsOf'].iloc[0]-pd.Timedelta(10, unit='day'), month_df['AsOf'].iloc[-1]+pd.Timedelta(10, unit='day')])

plt.legend(loc='upper left')